# 02 · Stance Classifier Training

End-to-end walkthrough of training the immigration-law stance classifier. This notebook is designed to run on free Colab (T4 GPU) in roughly 30 minutes for 4 epochs.

**Goal:** Learn a multi-output regressor that scores legal text on four orthogonal axes:
1. Restrictiveness
2. Enforcement intensity
3. Legalization stance
4. Humanitarian framing

**Architecture:** DistilBERT shared encoder + 4 sigmoid regression heads.

**Why DistilBERT?** Smaller (~66M params), faster, fits easily on free Colab. Drop-in upgrade to `roberta-base` or `nlpaueb/legal-bert-base-uncased` if compute allows.

## 0. Environment setup

On Colab, run:
```bash
!pip install -q transformers datasets evaluate accelerate pandas pyarrow
```

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'src'))

import numpy as np
import pandas as pd
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Synthetic training data

For demonstration, we generate a small synthetic dataset. **Replace this with your real labeled corpus** for actual training. The schema:
- `text` (str) — section-level chunk of legal text
- `restrictiveness` (float in [0, 1])
- `enforcement` (float in [0, 1])
- `legalization` (float in [0, 1])
- `humanitarian` (float in [0, 1])

In [ ]:
# Tiny synthetic dataset just to make the notebook runnable end-to-end.
# Replace with real labels for production training.
synthetic_data = [
    {'text': 'Section 287(g) authorizes state and local law enforcement to perform immigration enforcement functions.',
     'restrictiveness': 0.8, 'enforcement': 0.95, 'legalization': 0.1, 'humanitarian': 0.15},
    {'text': 'The Diversity Visa Program shall make available 50,000 visas annually to immigrants from underrepresented countries.',
     'restrictiveness': 0.2, 'enforcement': 0.1, 'legalization': 0.7, 'humanitarian': 0.55},
    {'text': 'Persons unlawfully present for more than one year shall be barred from re-entry for ten years.',
     'restrictiveness': 0.85, 'enforcement': 0.7, 'legalization': 0.05, 'humanitarian': 0.1},
    {'text': 'Refugees fleeing persecution on account of race, religion, nationality, membership in a particular social group, or political opinion may apply for asylum.',
     'restrictiveness': 0.25, 'enforcement': 0.2, 'legalization': 0.7, 'humanitarian': 0.95},
    {'text': 'Temporary Protected Status shall be granted to nationals of designated countries experiencing armed conflict or environmental disaster.',
     'restrictiveness': 0.3, 'enforcement': 0.3, 'legalization': 0.65, 'humanitarian': 0.85},
    {'text': 'Aliens entering without inspection shall be subject to expedited removal without further hearing.',
     'restrictiveness': 0.9, 'enforcement': 0.95, 'legalization': 0.05, 'humanitarian': 0.05},
    {'text': 'Lawful permanent residents may petition for the immigration of their spouses and unmarried children.',
     'restrictiveness': 0.3, 'enforcement': 0.15, 'legalization': 0.6, 'humanitarian': 0.5},
    {'text': 'The annual cap for H-1B specialty occupation workers shall not exceed 65,000 plus 20,000 for advanced-degree holders.',
     'restrictiveness': 0.55, 'enforcement': 0.3, 'legalization': 0.45, 'humanitarian': 0.3},
    {'text': 'Childhood arrivals brought to the United States before age 16 may apply for deferred action and work authorization.',
     'restrictiveness': 0.2, 'enforcement': 0.15, 'legalization': 0.85, 'humanitarian': 0.85},
    {'text': 'The Secretary may construct additional barriers along the southwest border as necessary to deter unlawful entry.',
     'restrictiveness': 0.85, 'enforcement': 0.9, 'legalization': 0.05, 'humanitarian': 0.1},
    {'text': 'Persons granted asylum may apply for adjustment of status to lawful permanent resident after one year.',
     'restrictiveness': 0.25, 'enforcement': 0.15, 'legalization': 0.85, 'humanitarian': 0.8},
    {'text': 'Employer sanctions shall apply to any person who knowingly hires an alien unauthorized to work in the United States.',
     'restrictiveness': 0.7, 'enforcement': 0.8, 'legalization': 0.3, 'humanitarian': 0.25},
]

df = pd.DataFrame(synthetic_data)
print(f'Dataset size: {len(df)}')
df.head()

In [ ]:
# Train / val split. With real data (~500 chunks), use 80/10/10 train/val/test.
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.25, random_state=42)

# Save to parquet so the training script picks them up
Path('../data/processed').mkdir(parents=True, exist_ok=True)
train_df.to_parquet('../data/processed/stance_train.parquet', index=False)
val_df.to_parquet('../data/processed/stance_val.parquet', index=False)
print(f'Train: {len(train_df)} | Val: {len(val_df)}')

## 2. Train the model

Either run the CLI directly:

```bash
python -m migration_atlas.models.stance_classifier train --config configs/stance.yaml
```

Or call the training function inline below (better for Colab where we want to see logs):

In [ ]:
from migration_atlas.models.stance_classifier import StanceConfig, train_model

cfg = StanceConfig(
    base_model='distilbert-base-uncased',
    num_epochs=2,        # bumped up for real corpora; 2 is plenty for the 9-row toy set
    batch_size=4,
    train_path='../data/processed/stance_train.parquet',
    val_path='../data/processed/stance_val.parquet',
    output_dir='../checkpoints/stance-distilbert',
)
train_model(cfg)

## 3. Test inference

With a real corpus, MAE on each axis should land around 0.10–0.15 after 4 epochs. The toy set above won't generalize — it's a smoke test that the pipeline runs.

In [ ]:
from migration_atlas.models.stance_classifier import load_model, predict_stance

model, tokenizer, cfg = load_model('../checkpoints/stance-distilbert')

test_text = 'The Secretary shall expand expedited removal authority to all persons apprehended within 100 miles of the border.'
scores = predict_stance(test_text, model, tokenizer, cfg)
for axis, s in scores.items():
    print(f'  {axis:18s} {s:.3f}')

## 4. What to do next

1. **Build a real labeled corpus.** Aim for ~500 section-level chunks across federal and state immigration laws, double-coded by two annotators.
2. **Report inter-rater reliability** (Cohen's κ per axis) in the case-study writeup. Below 0.6 means the rubric needs sharpening.
3. **Try Legal-BERT.** `nlpaueb/legal-bert-base-uncased` is pre-trained on legal text and typically improves MAE by 0.02–0.04.
4. **Push to HuggingFace Hub** so others can use the trained model without retraining.